<a href="https://colab.research.google.com/github/Sabinx87/Advent/blob/main/transformer_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import math


'2.11.0+cpu'

In [ ]:
##our vocab
vocab={
    "the":1,
    "dog":2,
    "has":3,
    "tail":4,
}


In [ ]:
##embedding the word
vocab_size=4
embedding_dim=8
embedding=nn.Embedding(vocab_size,embedding_dim)
embedding.weight

Parameter containing:
tensor([[ 0.9345, -0.2302,  0.3648, -0.3094, -1.2305,  0.3777,  0.9193,  1.2284],
        [-1.2674, -1.6152, -0.2195,  0.6260, -0.2531, -0.4897,  0.0124,  0.4792],
        [ 0.0600, -1.5703,  0.3757,  0.3112,  0.1758,  0.4727, -1.8655,  0.0590],
        [ 1.3605, -0.4970,  0.1070, -1.2087, -2.6000,  0.7981, -0.0214,  1.2210]],
       requires_grad=True)

In [ ]:
token=torch.tensor([0,1,2,3])
x=embedding(token)
x[0]


tensor([ 0.9345, -0.2302,  0.3648, -0.3094, -1.2305,  0.3777,  0.9193,  1.2284],
       grad_fn=<SelectBackward0>)

In [ ]:
max_seq_length=10
embedding_dim=8
positional_embedding=nn.Embedding(max_seq_length,embedding_dim)
positions=torch.arange(4)

pos_vector=positional_embedding(positions)
pos_vector.shape


torch.Size([4, 8])

In [ ]:
x=x+pos_vector ##Adding token embedding and positional embedding

In [ ]:
##Q,K,V
d_model=8

W_Q=nn.Parameter(torch.randn(d_model,d_model))
W_K=nn.Parameter(torch.randn(d_model,d_model))
W_V=nn.Parameter(torch.randn(d_model,d_model))

Q=x @ W_Q
k=x @ W_K
V=x @ W_V


In [ ]:
##Calculation the attention score

score= Q @ k.T

score

tensor([[-25.3598,  45.0615,  30.7776,  31.1650],
        [ 48.8483, -42.7789, -38.3655,  81.2914],
        [-40.0997,  28.7171, -69.6837,  10.1075],
        [  0.9022,  29.5822, -41.4001,  27.1610]], grad_fn=<MmBackward0>)

In [ ]:
score = score / math.sqrt(d_model)
score

tensor([[-3.1700,  5.6327,  3.8472,  3.8956],
        [ 6.1060, -5.3474, -4.7957, 10.1614],
        [-5.0125,  3.5896, -8.7105,  1.2634],
        [ 0.1128,  3.6978, -5.1750,  3.3951]], grad_fn=<DivBackward0>)

In [ ]:
##casual masking

score = score.masked_fill(mask == 0, float('-inf'))
score


tensor([[-3.1700,    -inf,    -inf,    -inf],
        [ 6.1060, -5.3474,    -inf,    -inf],
        [-5.0125,  3.5896, -8.7105,    -inf],
        [ 0.1128,  3.6978, -5.1750,  3.3951]], grad_fn=<MaskedFillBackward0>)

In [ ]:
##Applying softmax

attention_weight=torch.softmax(score, dim=-1)
attention_weight

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [9.9999e-01, 1.0613e-05, 0.0000e+00, 0.0000e+00],
        [1.8368e-04, 9.9981e-01, 4.5504e-06, 0.0000e+00],
        [1.5700e-02, 5.6602e-01, 7.9328e-05, 4.1820e-01]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
contextual_embedding=attention_weight @ V
contextual_embedding

tensor([[-9.6216,  0.5277,  1.2929, -7.0567, -6.2544,  5.3139, -5.6076,  0.1996],
        [-9.6214,  0.5277,  1.2929, -7.0566, -6.2544,  5.3138, -5.6076,  0.1995],
        [ 2.2269, -2.9298, -5.9021,  2.5232,  1.2727, -7.0925, -3.6374, -6.4602],
        [ 1.1016, -3.3513, -3.0603,  1.4884,  1.4864, -3.0824, -2.5114, -5.0734]],
       grad_fn=<MmBackward0>)

In [ ]:
###multi head attention
num_heads = 2
d_model = 8
seq_len = 4

head_dim = d_model // num_heads

# Q, K, V
W_Q = nn.Parameter(torch.randn(d_model, d_model))
W_K = nn.Parameter(torch.randn(d_model, d_model))
W_V = nn.Parameter(torch.randn(d_model, d_model))

Q = x @ W_Q
K = x @ W_K
V = x @ W_V

# Split into heads
Q = Q.view(seq_len, num_heads, head_dim).transpose(0, 1)
K = K.view(seq_len, num_heads, head_dim).transpose(0, 1)
V = V.view(seq_len, num_heads, head_dim).transpose(0, 1)

# Attention scores
scores = Q @ K.transpose(-2, -1)
scores = scores / math.sqrt(head_dim)

# Causal mask
mask = torch.tril(torch.ones(seq_len, seq_len))
scores = scores.masked_fill(mask == 0, float('-inf'))

# Attention weights
attention_weights = torch.softmax(scores, dim=-1)

# Contextual information
context = attention_weights @ V

# Combine heads
context = context.transpose(0, 1).contiguous()
context = context.view(seq_len, d_model)

# Output projection
W_O = nn.Parameter(torch.randn(d_model, d_model))
output = context @ W_O

output

tensor([[-10.5404,  -5.9005,  -1.8158,   4.1043,  10.1530,  14.8554,  -1.8556,
           1.4361],
        [ -7.8555,  -2.7478,   0.0233,   3.5607,   9.1443,   9.6759,  -2.4737,
          -3.3964],
        [ -6.8118,   3.2413,   4.7741,   5.1407,   8.6775,  -2.4932,  -7.1818,
         -13.3879],
        [-11.1969,  -2.6056,  -6.0466,   4.7128,   5.2706,   4.9604,  -2.0413,
         -10.7577]], grad_fn=<MmBackward0>)

In [ ]:
layer_norm = nn.LayerNorm(d_model)

add_norm_output = layer_norm(x + output)

add_norm_output.shape

torch.Size([4, 8])

In [ ]:
##FFb
##Weight and Bias


d_model=8
d_ff=32

w1=nn.Parameter(torch.randn(d_model,d_ff))
b1=nn.Parameter(torch.randn(d_ff))

w2=nn.Parameter(torch.randn(d_ff,d_model))
b2=nn.Parameter(torch.randn(d_model))

In [ ]:
##Fordward pass
ffn_hidden=add_norm_output @ w1 + b1
ffn_hidden=torch.relu(ffn_hidden)
ffn_output=ffn_hidden @ w2 + b2

ffn_output.shape

torch.Size([4, 8])

In [ ]:
##second residual connection
final_output = layer_norm(add_norm_output + ffn_output)

In [ ]:

#Output Projection
W_vocab=nn.Parameter(torch.randn(d_model,vocab_size))
b_vocab=nn.Parameter(torch.randn(vocab_size))

logits=final_output @ W_vocab + b_vocab
logits.shape

torch.Size([4, 4])

In [ ]:
probabilities= torch.softmax(logits, dim=-1)
probabilities

tensor([[8.9439e-01, 9.9407e-02, 1.5531e-04, 6.0441e-03],
        [7.1602e-01, 2.8039e-01, 3.5721e-04, 3.2390e-03],
        [7.1621e-01, 2.2206e-01, 1.1204e-02, 5.0527e-02],
        [9.5523e-02, 8.8153e-01, 1.4159e-03, 2.1530e-02]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
##creating input target

input_tokens = torch.tensor([0, 1, 2])
target_tokens = torch.tensor([1, 2, 3])

In [ ]:
tokens = torch.tensor([0, 1, 2, 3])
targets = torch.tensor([1, 2, 3, -100])



In [ ]:
#loss function
loss_function = nn.CrossEntropyLoss(ignore_index=-100)

In [ ]:
##optimizer
optimizer = torch.optim.Adam(
    [
        W_Q, W_K, W_V, W_O,
        w1, b1, w2, b2,
        W_vocab, b_vocab,
        embedding.weight,
        positional_embedding.weight,
        layer_norm.weight,
        layer_norm.bias
    ],
    lr=0.001
)

In [ ]:
for epoch in range(1000):

    # -------------------------
    # Forward pass
    # -------------------------

    # Token embeddings
    x = embedding(tokens)

    # Position embeddings
    positions = torch.arange(seq_len)
    pos_vectors = positional_embedding(positions)

    x = x + pos_vectors

    # Q, K, V
    Q = x @ W_Q
    K = x @ W_K
    V = x @ W_V

    # Split heads
    Q = Q.view(seq_len, num_heads, head_dim).transpose(0, 1)
    K = K.view(seq_len, num_heads, head_dim).transpose(0, 1)
    V = V.view(seq_len, num_heads, head_dim).transpose(0, 1)

    # Attention
    scores = Q @ K.transpose(-2, -1)
    scores = scores / math.sqrt(head_dim)

    # Causal mask
    mask = torch.tril(torch.ones(seq_len, seq_len))
    scores = scores.masked_fill(
        mask == 0,
        float('-inf')
    )

    # Attention probabilities
    attention_weights = torch.softmax(scores, dim=-1)

    # Context
    context = attention_weights @ V

    # Combine heads
    context = context.transpose(0, 1).contiguous()
    context = context.view(seq_len, d_model)

    # Output projection
    output = context @ W_O

    # First Add + Norm
    add_norm_output = layer_norm(x + output)

    # FFN
    ffn_hidden = add_norm_output @ w1 + b1
    ffn_hidden = torch.relu(ffn_hidden)

    ffn_output = ffn_hidden @ w2 + b2

    # Second Add + Norm
    final_output = layer_norm(
        add_norm_output + ffn_output
    )

    # Vocabulary projection
    logits = final_output @ W_vocab + b_vocab

    # -------------------------
    # Calculate loss
    # -------------------------

    loss = loss_function(
        logits,
        targets
    )

    # -------------------------
    # Backpropagation
    # -------------------------

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    # Print loss
    if epoch % 100 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item():.4f}"
        )

Epoch 0, Loss: 3.1866
Epoch 100, Loss: 0.2942
Epoch 200, Loss: 0.0783
Epoch 300, Loss: 0.0344
Epoch 400, Loss: 0.0199
Epoch 500, Loss: 0.0129
Epoch 600, Loss: 0.0091
Epoch 700, Loss: 0.0067
Epoch 800, Loss: 0.0051
Epoch 900, Loss: 0.0041


In [ ]:
with torch.no_grad():
    probabilities = torch.softmax(logits, dim=-1)

    predictions = torch.argmax(probabilities, dim=-1)

print("Predicted token IDs:", predictions)

Predicted token IDs: tensor([1, 2, 3, 1])


In [ ]:
with torch.no_grad():

    # Embedding
    x = embedding(input_tokens)

    # Position embedding
    positions = torch.arange(len(input_tokens))
    x = x + positional_embedding(positions)

    # Q, K, V
    Q = x @ W_Q
    K = x @ W_K
    V = x @ W_V

    # Split into heads
    seq_len = len(input_tokens)

    Q = Q.view(seq_len, num_heads, head_dim).transpose(0, 1)
    K = K.view(seq_len, num_heads, head_dim).transpose(0, 1)
    V = V.view(seq_len, num_heads, head_dim).transpose(0, 1)

    # Attention
    scores = Q @ K.transpose(-2, -1)
    scores = scores / math.sqrt(head_dim)

    # Causal mask
    mask = torch.tril(torch.ones(seq_len, seq_len))

    scores = scores.masked_fill(
        mask == 0,
        float('-inf')
    )

    # Softmax
    attention_weights = torch.softmax(scores, dim=-1)

    # Context
    context = attention_weights @ V

    # Combine heads
    context = context.transpose(0, 1).contiguous()
    context = context.view(seq_len, d_model)

    # Output projection
    output = context @ W_O

    # First Add + Norm
    add_norm_output = layer_norm(x + output)

    # FFN
    ffn_hidden = add_norm_output @ w1 + b1
    ffn_hidden = torch.relu(ffn_hidden)

    ffn_output = ffn_hidden @ w2 + b2

    # Second Add + Norm
    final_output = layer_norm(
        add_norm_output + ffn_output
    )

    # Vocabulary projection
    logits = final_output @ W_vocab + b_vocab

In [ ]:
last_logits = logits[-1]

In [ ]:
probabilities = torch.softmax(last_logits, dim=-1)

print(probabilities)

tensor([3.0772e-04, 2.9155e-03, 2.4591e-03, 9.9432e-01])


In [ ]:
predicted_id = torch.argmax(probabilities)
predicted_id

tensor(3)